<a href="https://colab.research.google.com/github/zia207/Causal_Inference_R/blob/main/Notebook/02_08_05_03_04_uplift_xgboost_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1O_99Xsf5TDRKoV8qWAuPhRoqeqlKSfFk)

# 5.3.4 Uplift XGBoost Regression

This tutorial applies **uplift XGBoost regression** from **RCausalML** to the **abortion** dataset from `{causaldata}`. The dataset features a **continuous** outcome (logged gonorrhea rate, `lnr`) and a **binary** treatment (early repeal of abortion prohibition, `repeal`). We will cover:

1. The conceptual foundations of T-learner uplift with gradient boosting
2. Data preparation and feature selection
3. Fitting uplift models with `uplift_xgboostRegressor()`
4. **Comparing** uplift XGBoost with `uplift_randomForestRegressor()` (T-learner and IT forests)
5. Evaluating uplift ranking via gain curves and AUUC
6. Interpreting heterogeneity in predicted CATE with kernel SHAP

> **Prerequisite:** install the optional **xgboost** package (`install.packages("xgboost")`).

## Overview

A standard predictive model asks: *"What outcome do we expect for this unit?"*

An **uplift model** asks a different question:

> *"How much would the outcome change **because of the treatment** for this unit, given their covariates?"*

That quantity is the **Conditional Average Treatment Effect (CATE)**. For a unit with covariates $X = x$ and a continuous outcome $Y$:

$$\tau(x) = \mathbb{E}[Y \mid T = 1, X = x] - \mathbb{E}[Y \mid T = 0, X = x]$$

where $T = 1$ is treated and $T = 0$ is control. **Heterogeneity** means $\tau(x)$ varies with $x$: some subgroups benefit strongly, others barely, and some may be harmed.

For **binary outcomes** (e.g. conversion yes/no), the CATE is:

$$\tau(x) = P(Y = 1 \mid T = 1, X = x) - P(Y = 1 \mid T = 0, X = x)$$

## Why gradient boosting for uplift?

`uplift_xgboost()` implements a **T-learner** with **XGBoost** as the base learner:

1. Fit one boosted tree ensemble on control units ($T = 0$) to estimate $\hat{\mu}_0(x)$.
2. Fit a separate ensemble on treated units ($T = 1$) to estimate $\hat{\mu}_1(x)$.
3. Predict CATE as the difference:

$$\hat{\tau}(x) = \hat{\mu}_1(x) - \hat{\mu}_0(x)$$

Compared with uplift **random forests** (bagged uplift trees with treatment-aware splits), uplift XGBoost:

| Aspect | Uplift random forest | Uplift XGBoost |
|--------|----------------------|----------------|
| Base learner | Bagged uplift / `ranger` trees | Gradient-boosted trees (`xgboost`) |
| Splitting rule | KL / ED / IT / T-learner `ranger` | T-learner only |
| Strengths | Interpretable tree ensembles; IT/CIT forests | Flexible nonlinear surfaces; strong predictive performance |
| `RCausalML` API | `uplift_randomForestRegressor()` | `uplift_xgboostRegressor()` |

The model output is $\hat{\tau}(x)$ **on the scale of $Y$**. In this notebook, $Y$ is `lnr` (log gonorrhea rate per 100,000), so a CATE of $-0.12$ means the model estimates that early repeal is associated with a **0.12 log-point lower** gonorrhea rate for that covariate profile — not that the raw rate drops by 12%.

#### From ATE to targeted policy

The **average treatment effect (ATE)** summarises the treatment in one number. An uplift model estimates the **full profile** $\tau(x)$, enabling:

| CATE region | Interpretation | Targeting decision |
|-------------|----------------|-------------------|
| **High positive** $\tau(x)$ | Large expected benefit from treatment | Priority targets |
| **Near zero** | Treatment adds little | Skip / low priority |
| **Negative** $\tau(x)$ | Treatment may worsen outcomes | Actively avoid treating |

Standard outcome prediction ranks high-baseline units highly even when treatment adds nothing. Uplift models rank by **incremental effect**, directly supporting **personalized treatment rules**.

#### Binary vs multi-arm treatment

- **Binary** ($T \in \{0,1\}$): one T-learner pair estimates control-vs-treated uplift.
- **Multi-arm**: `RCausalML` fits **one model pair per non-control arm**, returning a vector of uplifts and a recommended arm.

The **abortion** dataset has a **single** policy indicator (`repeal`), so this notebook uses the binary path throughout.

## Setup R in Python Runtime - Install {rpy2}

{rpy2} allows running R code in Colab Python runtime via `%%R` magic.

In [ ]:
!pip uninstall rpy2 -y

!pip install rpy2==3.5.1

%load_ext rpy2.ipython


## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Set Up

### Check and Install Required R Packages

Following R packages are required to run this notebook. If any of these packages are not installed, you can install them using the code below:

`tidyverse`, `RCausalML`, `causaldata`, `xgboost`, `rpart.plot`, `shapviz`, `kernelshap`

In [ ]:
%%R
packages <- c(
  "tidyverse",
  "RCausalML",
  "causaldata",
  "xgboost",
  "rpart.plot",
  "shapviz",
  "kernelshap"
)


### Install Missing Packages

In [ ]:
%%R
# Install missing packages
# new_packages <- packages[!(packages %in% installed.packages()[, "Package"])]
# if (length(new_packages)) install.packages(new_packages)


### Verify Installation

In [ ]:
%%R
# Verify installation
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))


### Load R Packages

In [ ]:
%%R
# Load packages with suppressed messages
invisible(lapply(packages, function(pkg) {
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
})


### Check Loaded Packages

In [ ]:
%%R
# Check loaded packages
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])


In [ ]:
%%R
if (!requireNamespace("xgboost", quietly = TRUE)) {
  stop("This notebook requires the xgboost package: install.packages(\"xgboost\")")
}


### Data

The `{causaldata}` package provides curated datasets for causal inference. We load the **abortion** dataset, which records state-year observations on gonorrhea rates among 15–19 year-olds, alongside a binary indicator for early repeal of abortion prohibition. The dataset is commonly used to study how abortion legalisation policies affected risky sexual behaviour among teenagers, proxied by gonorrhea incidence.

**Key variables:**

| Variable | Description |
|----------|-------------|
| `fip` | State FIPS code |
| `age` | Age in years (focus: 15–19) |
| `race` | Race (1 = white, 2 = black) |
| `sex` | Sex (1 = male, 2 = female) |
| `year` | Calendar year |
| `t` | Rescaled time variable |
| `totpop` | Total population (state-level) |
| `ir` | Incarcerated males per 100,000 |
| `crack` | Crack cocaine index |
| `alcohol` | Alcohol consumption per capita |
| `income` | Real income per capita |
| `ur` | State unemployment rate |
| `poverty` | Poverty rate |
| **`repeal`** | **Treatment**: 1 = early repeal of abortion prohibition, 0 = otherwise |
| **`lnr`** | **Outcome**: log gonorrhea rate per 100,000 (15–19 year-olds) |
| `pi` | Parental involvement law in effect |
| `bf15` | Indicator: black female aged 15–19 |

> **Causal note:** This is observational panel data. Uplift models estimate *conditional contrasts* $\hat{\tau}(x)$, not causal effects per se. Causal interpretation depends on your identification assumptions (e.g. fixed effects, difference-in-differences).

In [ ]:
%%R
list_causaldata_datasets()


### Load and inspect

In [ ]:
%%R
abortion <- load_causaldata("abortion")$data

head(abortion)
str(abortion)
summary(abortion$repeal)           # Treatment balance
table(abortion$repeal, abortion$race)  # Example crosstab


### Data processing

We restrict to the 15–19 age group and drop rows with any missing values, then define covariate matrix $X$, outcome $Y$, and treatment $W$.

In [ ]:
%%R
abortion_clean <- abortion |>
  dplyr::filter(age >= 15 & age <= 19) |>
  na.omit()

# Define X, Y, W
X <- abortion_clean[, c("age", "race", "sex", "totpop", "ir", "crack",
                         "alcohol", "income", "ur", "poverty")]
Y <- abortion_clean$lnr    # Continuous outcome: log gonorrhea rate
W <- abortion_clean$repeal # Binary treatment: 1 = early repeal

# Convert X to numeric matrix and drop near-zero variance columns
X <- as.matrix(X)
X <- X[, apply(X, 2, stats::var) > 1e-10, drop = FALSE]
cat("Covariates used:", colnames(X), "\n")


### Train / test split

We use an 80 / 20 random split (seed fixed for reproducibility).

In [ ]:
%%R
set.seed(111)
n         <- nrow(abortion_clean)
train_idx <- sample(seq_len(n), size = round(0.8 * n))

X_train <- X[train_idx,  , drop = FALSE]
Y_train <- Y[train_idx]
W_train <- W[train_idx]

X_test  <- X[-train_idx, , drop = FALSE]
Y_test  <- Y[-train_idx]
W_test  <- W[-train_idx]

x_names <- colnames(X_train)

cat("Training rows:", nrow(X_train), "\n")
cat("Test rows:    ", nrow(X_test),  "\n")
cat("Columns:      ", x_names,       "\n")


Outcome by treatment arm:

In [ ]:
%%R
agg <- aggregate(
  lnr ~ repeal,
  data = data.frame(lnr = Y, repeal = W),
  FUN  = function(z) c(mean = mean(z), sd = stats::sd(z), size = length(z))
)
agg <- do.call(data.frame, agg)
names(agg) <- c("repeal", "mean", "sd", "size")
agg <- rbind(agg, data.frame(repeal = "All", mean = mean(Y),
                              sd = stats::sd(Y), size = length(Y)))
print(agg)


### Feature Selection (F-filter)

Following Zhao et al. (2020), we use the **F-filter** (`get_importance(..., method = "F")`) to rank covariates by their treatment–feature interaction strength, measured via F-tests in linear models. This is the appropriate filter for continuous outcomes like `lnr`.

> **Why not `LR` or `KL`?** Those filters in `RCausalML` are implemented for **binary** outcomes only. For continuous outcomes, use `method = "F"`.
>
> **Training-only rule:** feature ranking uses the *training set only* to avoid data leakage into model evaluation.

## Rank features by F-test (orders 1–3)

In [ ]:
%%R
df_fs_train <- as.data.frame(X_train, stringsAsFactors = FALSE)
names(df_fs_train) <- x_names
df_fs_train$lnr <- Y_train
df_fs_train$treatment_group_key <- ifelse(W_train == 1, "treated", "control")


In [ ]:
%%R
f_imp_1 <- get_importance(
  df_fs_train,
  features                  = x_names,
  y_name                    = "lnr",
  method                    = "F",
  experiment_group_column   = "treatment_group_key",
  control_group             = "control",
  treatment_group           = "treated",
  order                     = 1
)

f_imp_2 <- get_importance(
  df_fs_train,
  features                  = x_names,
  y_name                    = "lnr",
  method                    = "F",
  experiment_group_column   = "treatment_group_key",
  control_group             = "control",
  treatment_group           = "treated",
  order                     = 2
)

f_imp_3 <- get_importance(
  df_fs_train,
  features                  = x_names,
  y_name                    = "lnr",
  method                    = "F",
  experiment_group_column   = "treatment_group_key",
  control_group             = "control",
  treatment_group           = "treated",
  order                     = 3
)

cat("*** Order 1 ***\n"); head(f_imp_1)
cat("*** Order 2 ***\n"); head(f_imp_2)
cat("*** Order 3 ***\n"); head(f_imp_3)


#### Feature importance plot

In [ ]:
%%R
par(mar = c(4, 8, 2, 1))
n_bar <- min(20L, nrow(f_imp_1))
barplot(
  rev(head(f_imp_1$score, n_bar)),
  names.arg = rev(head(f_imp_1$feature, n_bar)),
  horiz     = TRUE,
  las       = 1,
  main      = "F1 filter: feature importance (abortion, train)",
  col       = "steelblue",
  border    = NA
)


#### Define feature subsets

In [ ]:
%%R
ranked_f     <- f_imp_1$feature
n_f          <- length(ranked_f)
top_3_features <- ranked_f[seq_len(min(3L, n_f))]
top_6_features <- ranked_f[seq_len(min(6L, n_f))]

cat("Top 3 (F1):", paste(top_3_features, collapse = ", "), "\n")
cat("Top 6 (F1):", paste(top_6_features, collapse = ", "), "\n")


### Comparing Learners across Feature Subsets

We compare **cumulative gain** and **AUUC** (area under the uplift curve, normalised) on the test set for T-, R-, and S-learners fitted on all features, the top-6 subset, and the top-3 subset. Here we use **XGBoost** in the T-learner to align with the uplift XGBoost model below.

In [ ]:
%%R
is_treated  <- as.integer(W_test)
y_test      <- Y_test
n_test      <- nrow(X_test)
model_cols  <- c("All", "Top_3", "Top_6", "Random")


### T-Learner (XGBoost)

In [ ]:
%%R
fit_tl <- function(feats) {
  m <- TLearner(learner = "xgb", control_name = 0)
  m <- fit(m, X_train[, feats, drop = FALSE], W_train, Y_train,
           nrounds = 80, max_depth = 4)
  p <- predict(m, X_test[, feats, drop = FALSE])
  if (is.matrix(p)) p[, 1] else p
}

pred_tl_all <- fit_tl(x_names)
pred_tl_t3  <- fit_tl(top_3_features)
pred_tl_t6  <- fit_tl(top_6_features)


In [ ]:
%%R
set.seed(42)
df_tl <- data.frame(
  All        = pred_tl_all,
  Top_3      = pred_tl_t3,
  Top_6      = pred_tl_t6,
  Random     = stats::runif(n_test),
  is_treated = is_treated,
  lnr        = y_test
)
plot_gain(df_tl,
          outcome_col   = "lnr",
          treatment_col = "is_treated",
          model_cols    = model_cols,
          main          = "T-Learner (XGBoost): cumulative gain — abortion test set")

gain_tl  <- get_cumgain(df_tl, outcome_col = "lnr",
                         treatment_col = "is_treated", normalize = TRUE)
auuc_tl  <- colMeans(gain_tl[, model_cols, drop = FALSE])
print(round(auuc_tl, 4))


### R-Learner (ranger baseline)

In [ ]:
%%R
fit_rl <- function(feats) {
  m <- RLearner(learner = "ranger", control_name = 0, n_fold = 3)
  m <- fit(m, X_train[, feats, drop = FALSE], W_train, Y_train)
  p <- predict(m, X_test[, feats, drop = FALSE])
  if (is.matrix(p)) p[, 1] else p
}

pred_rl_all <- fit_rl(x_names)
pred_rl_t3  <- fit_rl(top_3_features)
pred_rl_t6  <- fit_rl(top_6_features)


In [ ]:
%%R
set.seed(42)
df_rl <- data.frame(
  All        = pred_rl_all,
  Top_3      = pred_rl_t3,
  Top_6      = pred_rl_t6,
  Random     = stats::runif(n_test),
  is_treated = is_treated,
  lnr        = y_test
)
plot_gain(df_rl,
          outcome_col   = "lnr",
          treatment_col = "is_treated",
          model_cols    = model_cols,
          main          = "R-Learner (ranger): cumulative gain — abortion test set")

gain_rl  <- get_cumgain(df_rl, outcome_col = "lnr",
                         treatment_col = "is_treated", normalize = TRUE)
auuc_rl  <- colMeans(gain_rl[, model_cols, drop = FALSE])
print(round(auuc_rl, 4))


### S-Learner (ranger baseline)

In [ ]:
%%R
fit_sl <- function(feats) {
  m <- SLearner(learner = "ranger", control_name = 0)
  m <- fit(m, X_train[, feats, drop = FALSE], W_train, Y_train)
  p <- predict(m, X_test[, feats, drop = FALSE])
  if (is.matrix(p)) p[, 1] else p
}

pred_sl_all <- fit_sl(x_names)
pred_sl_t3  <- fit_sl(top_3_features)
pred_sl_t6  <- fit_sl(top_6_features)


In [ ]:
%%R
set.seed(42)
df_sl <- data.frame(
  All        = pred_sl_all,
  Top_3      = pred_sl_t3,
  Top_6      = pred_sl_t6,
  Random     = stats::runif(n_test),
  is_treated = is_treated,
  lnr        = y_test
)
plot_gain(df_sl,
          outcome_col   = "lnr",
          treatment_col = "is_treated",
          model_cols    = model_cols,
          main          = "S-Learner (ranger): cumulative gain — abortion test set")

gain_sl  <- get_cumgain(df_sl, outcome_col = "lnr",
                         treatment_col = "is_treated", normalize = TRUE)
auuc_sl  <- colMeans(gain_sl[, model_cols, drop = FALSE])
print(round(auuc_sl, 4))


### AUUC summary table

In [ ]:
%%R
auuc_tab <- rbind(
  data.frame(Learner = "T-Learner (XGBoost)", Feature_set = names(auuc_tl),
             AUUC = as.numeric(auuc_tl)),
  data.frame(Learner = "R-Learner (ranger)", Feature_set = names(auuc_rl),
             AUUC = as.numeric(auuc_rl)),
  data.frame(Learner = "S-Learner (ranger)", Feature_set = names(auuc_sl),
             AUUC = as.numeric(auuc_sl))
)
print(round(auuc_tab, 4))


> **Reading the table:** Higher AUUC means better uplift ranking. Smaller feature sets can match or beat "all features" when the dropped covariates are noisy — this is the practical payoff of filter-based feature selection.

## Uplift XGBoost with `uplift_xgboostRegressor()`

The unified uplift API in `RCausalML` for gradient boosting. `uplift_xgboost()` fits separate XGBoost models per treatment arm and returns CATE as the predicted outcome difference.

> **Note:** Only `evaluation_function = "TLearner"` is supported. Tree-based uplift split criteria (KL, IT, etc.) are available in `uplift_randomForestRegressor()` instead.

### Fit T-learner uplift XGBoost

In [ ]:
%%R
uplift_model <- uplift_xgboostRegressor(
  X_train, W_train, Y_train,
  n_estimators        = 100,
  max_depth           = 4,
  evaluation_function = "TLearner",
  random_state        = 111,
  learning_rate       = 0.05,
  subsample           = 0.8
)

pred_tr <- predict(uplift_model, X_train)
pred_te <- predict(uplift_model, X_test)

cat("Mean predicted CATE — train:", round(mean(pred_tr), 4),
    " | test:", round(mean(pred_te), 4), "\n")


The `full_output = TRUE` argument returns the control arm prediction, treatment arm prediction, and CATE (delta) for each unit — useful for auditing and downstream analysis.

In [ ]:
%%R
pred_full <- predict(uplift_model, X_test, full_output = TRUE)
head(pred_full)


Model summary:

In [ ]:
%%R
print(uplift_model)


## Comparing `uplift_xgboostRegressor()` vs `uplift_randomForestRegressor()`

Both functions share the same uplift API in `RCausalML`: binary / multi-arm treatment, `predict(..., full_output = TRUE)`, and CATE on the outcome scale. The key difference is the base learner inside each arm:

| Model | Function | T-learner base | Uplift-specific splits |
|-------|----------|----------------|------------------------|
| Uplift XGBoost | `uplift_xgboostRegressor()` | XGBoost per arm | Not available (`TLearner` only) |
| Uplift random forest | `uplift_randomForestRegressor()` | `ranger` per arm (`evaluation_function = "TLearner"`) | IT / CIT bagged uplift trees |

We fit all three on the **same train/test split** (seed `111`) so uplift ranking metrics are directly comparable.

### T-learner random forest

In [ ]:
%%R
uplift_rf_tl <- uplift_randomForestRegressor(
  X_train, W_train, Y_train,
  n_estimators        = 100,
  min_samples_leaf    = 80,
  max_depth           = 8,
  evaluation_function = "TLearner",
  random_state        = 111
)

pred_rf_tl_tr <- predict(uplift_rf_tl, X_train)
pred_rf_tl_te <- predict(uplift_rf_tl, X_test)

cat("Uplift RF (TLearner) — mean CATE train:", round(mean(pred_rf_tl_tr), 4),
    " | test:", round(mean(pred_rf_tl_te), 4), "\n")


In [ ]:
%%R
pred_rf_tl_full <- predict(uplift_rf_tl, X_test, full_output = TRUE)
head(pred_rf_tl_full)


### Interaction-tree random forest

The random forest also supports treatment-effect-aware splitting via interaction trees (`evaluation_function = "IT"`), which uplift XGBoost does not provide.

In [ ]:
%%R
uplift_rf_it <- uplift_randomForestRegressor(
  X_train, W_train, Y_train,
  n_estimators        = 50,
  min_samples_leaf    = 80,
  max_depth           = 4,
  evaluation_function = "IT",
  random_state        = 111
)

pred_rf_it_te <- predict(uplift_rf_it, X_test)
cat("Uplift RF (IT) — mean CATE (test):", round(mean(pred_rf_it_te), 4), "\n")


In [ ]:
%%R
print(uplift_rf_tl)
print(uplift_rf_it)


### Mean CATE and prediction correlation

In [ ]:
%%R
cmp_mean <- data.frame(
  Model = c(
    "Uplift XGBoost (TLearner)",
    "Uplift RF (TLearner)",
    "Uplift RF (IT)"
  ),
  Mean_CATE_train = c(mean(pred_tr), mean(pred_rf_tl_tr), NA_real_),
  Mean_CATE_test  = c(mean(pred_te), mean(pred_rf_tl_te), mean(pred_rf_it_te))
)
print(round(cmp_mean, 4))

corr_tab <- data.frame(
  Pair = c("XGBoost vs RF (TLearner)", "XGBoost vs RF (IT)", "RF (TLearner) vs RF (IT)"),
  Correlation = c(
    stats::cor(pred_te, pred_rf_tl_te),
    stats::cor(pred_te, pred_rf_it_te),
    stats::cor(pred_rf_tl_te, pred_rf_it_te)
  )
)
print(round(corr_tab, 4))


In [ ]:
%%R
par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
plot(pred_rf_tl_te, pred_te,
     xlab = "Uplift RF (TLearner)", ylab = "Uplift XGBoost",
     main = "Predicted CATE — test set",
     pch = 19, col = rgb(0.2, 0.4, 0.8, 0.5))
abline(0, 1, lty = 2, col = "gray40")
plot(pred_rf_it_te, pred_te,
     xlab = "Uplift RF (IT)", ylab = "Uplift XGBoost",
     main = "Predicted CATE — test set",
     pch = 19, col = rgb(0.8, 0.4, 0.2, 0.5))
abline(0, 1, lty = 2, col = "gray40")
par(mfrow = c(1, 1))


## Evaluation

### Gain curve and AUUC on the test set

The **cumulative gain curve** plots, as we move through units in descending order of predicted CATE, how much *actual* treatment benefit we capture relative to random assignment. A curve bowing above the diagonal indicates useful uplift ranking.

In [ ]:
%%R
df_gain <- data.frame(
  is_treated   = W_test,
  outcome      = Y_test,
  uplift_xgb   = pred_te,
  uplift_rf_tl = pred_rf_tl_te,
  uplift_rf_it = pred_rf_it_te,
  Random       = stats::runif(nrow(X_test))
)
head(df_gain)


In [ ]:
%%R
uplift_cols <- c("uplift_xgb", "uplift_rf_tl", "uplift_rf_it", "Random")
plot_gain(
  df_gain,
  outcome_col   = "outcome",
  treatment_col = "is_treated",
  model_cols    = uplift_cols,
  main          = "Uplift XGBoost vs uplift RF — abortion test set"
)

gain_uplift <- get_cumgain(
  df_gain,
  outcome_col   = "outcome",
  treatment_col = "is_treated",
  normalize     = TRUE
)
auuc_uplift <- colMeans(gain_uplift[, uplift_cols, drop = FALSE])
auuc_uplift_tab <- data.frame(
  Model = uplift_cols,
  AUUC  = as.numeric(auuc_uplift)
)
print(round(auuc_uplift_tab, 4))


> **Reading AUUC:** higher is better uplift ranking. T-learner XGBoost and T-learner RF both estimate $\hat{\mu}_1(x) - \hat{\mu}_0(x)$ with different base learners; the IT forest optimises treatment–covariate interaction splits and may rank units differently even when mean CATE is similar.

#### Distribution of predicted CATE

Fitting on the **full dataset** compares how each model spreads heterogeneity across all observed covariate profiles.

In [ ]:
%%R
ux_full <- uplift_xgboostRegressor(
  X, W, Y,
  n_estimators        = 80,
  max_depth           = 4,
  evaluation_function = "TLearner",
  random_state        = 111,
  learning_rate       = 0.05
)
uf_tl_full <- uplift_randomForestRegressor(
  X, W, Y,
  n_estimators        = 80,
  min_samples_leaf    = 40,
  max_depth           = 8,
  evaluation_function = "TLearner",
  random_state        = 111
)
uf_it_full <- uplift_randomForestRegressor(
  X, W, Y,
  n_estimators        = 50,
  min_samples_leaf    = 40,
  max_depth           = 4,
  evaluation_function = "IT",
  random_state        = 111
)

pred_cate_xgb  <- predict(ux_full, X)
pred_cate_rf_tl <- predict(uf_tl_full, X)
pred_cate_rf_it <- predict(uf_it_full, X)


In [ ]:
%%R
hist(
  pred_cate_xgb,
  breaks = 30,
  col    = rgb(0.2, 0.4, 0.8, 0.45),
  border = NA,
  main   = "Predicted CATE distributions (full sample)",
  xlab   = "Predicted treatment effect (lnr scale)"
)
hist(pred_cate_rf_tl, breaks = 30, col = rgb(0.2, 0.7, 0.3, 0.35), border = NA, add = TRUE)
hist(pred_cate_rf_it, breaks = 30, col = rgb(0.9, 0.5, 0.1, 0.35), border = NA, add = TRUE)
legend("topright",
       legend = c(
         sprintf("XGBoost (mean = %.3f)", mean(pred_cate_xgb)),
         sprintf("RF T-Learner (mean = %.3f)", mean(pred_cate_rf_tl)),
         sprintf("RF IT (mean = %.3f)", mean(pred_cate_rf_it))
       ),
       fill = c(rgb(0.2, 0.4, 0.8, 0.7), rgb(0.2, 0.7, 0.3, 0.7), rgb(0.9, 0.5, 0.1, 0.7)),
       bty = "n")


### Variable importance: XGBoost vs ranger (T-learner arms)

For **fair comparison of drivers**, we extract importance from the T-learner arm models and average across control / treated fits.

In [ ]:
%%R
imp0 <- xgboost::xgb.importance(model = uplift_model$fit_0)
imp1 <- xgboost::xgb.importance(model = uplift_model$fit_1)

avg_imp <- merge(
  imp0[, c("Feature", "Gain")],
  imp1[, c("Feature", "Gain")],
  by = "Feature",
  suffixes = c("_0", "_1"),
  all = TRUE
)
avg_imp$Gain_avg <- rowMeans(avg_imp[, c("Gain_0", "Gain_1")], na.rm = TRUE)
avg_imp <- avg_imp[order(-avg_imp$Gain_avg), ]

print(head(avg_imp, 8))


In [ ]:
%%R
imp_rf0 <- uplift_rf_tl$fit_0$variable.importance
imp_rf1 <- uplift_rf_tl$fit_1$variable.importance
imp_rf_avg <- sort((imp_rf0 + imp_rf1) / 2, decreasing = TRUE)
print(round(head(imp_rf_avg, 8), 4))


In [ ]:
%%R
top_n <- 6L
xgb_top <- head(avg_imp, top_n)
rf_top  <- head(imp_rf_avg, top_n)

par(mfrow = c(1, 2), mar = c(4, 8, 3, 1))
barplot(rev(xgb_top$Gain_avg), names.arg = rev(xgb_top$Feature),
        horiz = TRUE, las = 1, col = "darkorange", border = NA,
        main = "Uplift XGBoost\n(average gain)")
barplot(rev(rf_top), names.arg = rev(names(rf_top)),
        horiz = TRUE, las = 1, col = "forestgreen", border = NA,
        main = "Uplift RF T-Learner\n(average ranger impurity)")
par(mfrow = c(1, 1))


## Interpretable Single-Tree Models

Boosted ensembles are powerful but opaque. A single **interaction tree** or **causal inference tree** summarises learned heterogeneity in human-readable split rules, useful for communicating findings to non-technical stakeholders.

### Interaction tree (IT)

In [ ]:
%%R
it_fit <- interaction_tree(
  X_train, W_train, Y_train,
  min_node_size = 200,
  max_depth     = 4
)
# Print split rules
cat(uplift_tree_string(it_fit$tree, it_fit$X_names))


Leaf labels show $\hat{\tau} \approx \mathbb{E}[Y \mid T=1] - \mathbb{E}[Y \mid T=0]$ within each region.

In [ ]:
%%R
uplift_tree_plot(
  it_fit,
  main = "Interaction tree — lnr vs repeal (abortion data, train)"
)


### Causal inference tree (CIT)

In [ ]:
%%R
cit_fit <- causal_inference_tree(
  X_train, W_train, Y_train,
  min_node_size = 200,
  max_depth     = 4
)
uplift_tree_plot(
  cit_fit,
  main = "Causal inference tree (CIT) — abortion data (train)"
)


## Explaining Predicted CATE with Kernel SHAP

SHAP (SHapley Additive exPlanations) decomposes each unit's predicted CATE into additive contributions from each covariate. This shows *why* the model predicts high or low treatment benefit for a given unit.

> **Important:** SHAP explains heterogeneity in $\hat{\tau}(X)$, **not** heterogeneity in the raw outcome $\mathbb{E}[Y \mid X]$. A positive SHAP value for a feature means that feature pushes the *predicted treatment effect* up.
>
> Kernel SHAP is model-agnostic: it uses the model's `predict()` function and a reference background dataset. It can be slow for large datasets — we use a 100-row explain set and a 100-row background.

### Compute SHAP values (training set subset)

In [ ]:
%%R
set.seed(2026)

n_explain <- min(100L, nrow(X_train))
bg_size   <- min(100L, nrow(X_train))
X_explain <- X_train[seq_len(n_explain), , drop = FALSE]
bg_idx    <- sample.int(nrow(X_train), bg_size)
bg_X      <- X_train[bg_idx, , drop = FALSE]

pred_fun <- function(object, newdata) as.vector(predict(object, newdata))

library(future)
plan(multisession, workers = max(1L, parallel::detectCores() - 1L))

shap_ksh <- kernelshap(
  object   = uplift_model,
  X        = X_explain,
  bg_X     = bg_X,
  pred_fun = pred_fun,
  bg_w     = NULL,
  verbose  = FALSE
)
plan(sequential)

shp_uplift <- shapviz(shap_ksh, X = X_explain)


### Global feature importance

In [ ]:
%%R
sv_importance(shp_uplift, kind = "beeswarm")  # Distribution of SHAP by feature
sv_importance(shp_uplift, show_numbers = TRUE) # Bar chart with mean |SHAP|


### Dependence plots

These show how a feature's SHAP value changes as the feature value changes — revealing the shape of the heterogeneity.

In [ ]:
%%R
important_vars <- x_names[seq_len(min(4L, length(x_names)))]
sv_dependence(shp_uplift, v = important_vars, share_y = TRUE)


Interaction between the top two features:

In [ ]:
%%R
if (length(x_names) >= 2) {
  sv_dependence(shp_uplift, v = x_names[[2]], color_var = x_names[[1]])
}


### Waterfall plot: highest-uplift unit

A waterfall plot decomposes one unit's predicted CATE into covariate contributions, starting from the baseline (mean prediction across background) and adding or subtracting each feature's SHAP value.

In [ ]:
%%R
preds_explain <- pred_fun(uplift_model, X_explain)
row_id_max    <- which.max(preds_explain)

sv_waterfall(shp_uplift, row_id = row_id_max) +
  ggplot2::ggtitle("SHAP waterfall — unit with highest estimated CATE")

sv_force(shp_uplift, row_id = row_id_max)


### SHAP on the test set

In [ ]:
%%R
n_te <- min(10L, nrow(X_test))
shap_test <- kernelshap(
  uplift_model,
  X        = X_test[seq_len(n_te), , drop = FALSE],
  bg_X     = bg_X,
  pred_fun = pred_fun,
  verbose  = FALSE
)
shp_test <- shapviz(shap_test, X = X_test[seq_len(n_te), , drop = FALSE])
sv_importance(shp_test, kind = "beeswarm")


## Summary

This tutorial demonstrated the full workflow for uplift XGBoost regression with `RCausalML` on a real observational dataset:

| Step | What we did |
|------|-------------|
| **Data** | Loaded `abortion` via `{causaldata}`; defined `lnr` (continuous), `repeal` (binary) |
| **Feature selection** | F-filter (`get_importance(..., method = "F")`) on training data only; compared top-3 / top-6 / all features |
| **Learner comparison** | T-Learner (XGBoost), R-/S-Learners (ranger) × feature subsets evaluated by AUUC on the test set |
| **Uplift XGBoost** | `uplift_xgboostRegressor()` with T-learner XGBoost per arm |
| **Uplift random forest** | `uplift_randomForestRegressor()` with T-learner `ranger` and IT forests; AUUC, correlation, and importance compared head-to-head |
| **Evaluation** | Gain curves, AUUC table, CATE distributions, XGBoost gain vs ranger impurity importance |
| **Single trees** | `interaction_tree()` and `causal_inference_tree()` for interpretable split rules |
| **SHAP** | Kernel SHAP via `{kernelshap}` + `{shapviz}` — importance, dependence, waterfall |

### Key reminders

- **Scale of CATE:** predictions are on the `lnr` (log) scale. A CATE of $-0.10$ means the model expects $0.10$ fewer log-points of gonorrhea rate — not a 10% reduction in the raw rate.
- **Observational data:** the model identifies *conditional contrasts*, not causal effects. Your identification strategy (e.g. DiD, fixed effects) governs causal interpretation.
- **XGBoost vs random forest:** `uplift_xgboost()` supports T-learner boosting only. Use `uplift_randomForestRegressor(..., evaluation_function = "TLearner")` for a directly comparable T-learner baseline, or `"IT"` / `"CIT"` when you want treatment-effect-aware tree splits.
- **Multi-arm treatment:** if your dataset has multiple treatment arms, use `uplift_xgboost(..., control_name = ...)`.

## Resources

- [Uplift Trees Example — Python CausalML docs](https://causalml.readthedocs.io/en/latest/examples/uplift_trees_with_synthetic_data.html)
- [Feature Selection for Uplift Modelling — Python CausalML docs](https://causalml.readthedocs.io/en/latest/examples/feature_selection.html)
- Zhao, Z., Feng, T., Garg, V., & Cakmak, M. (2020). [Feature Selection Methods for Uplift Modeling](https://arxiv.org/abs/2005.03447). *arXiv*.
- Gutierrez, P., & Gérardy, J.-Y. (2017). [Causal inference and uplift modelling: A review of the literature](http://proceedings.mlr.press/v67/gutierrez17a/gutierrez17a.pdf). *PMLR*.
- Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD*.